# S1 Staging ? Products

Apply explicit product types and validate the catalog while preserving all source rows, NULL bundle costs, and component definitions. Bundles are not expanded.


# 1. Imports & Configuration


In [ ]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "raw_data").is_dir() and (path / "s1_staging").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the FABINI project root from the current directory.")

RAW_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "s1_staging" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_FILE = "raw_products.csv"
OUTPUT_PATH = OUTPUT_DIR / "products.parquet"
EXPECTED_ROWS = 28
EXPECTED_BUNDLES = 4
EXPECTED_COLUMNS = ["sku", "product_name", "category", "unit_cost_czk", "is_bundle", "bundle_components"]
REQUIRED_COLUMNS = ["sku", "product_name", "category", "is_bundle"]


# 2. Load Raw Data

Read text first to validate the source values before casting.


In [ ]:
raw = pl.read_csv(RAW_DIR / SOURCE_FILE, infer_schema=False, null_values=[""])
if raw.columns != EXPECTED_COLUMNS or raw.height != EXPECTED_ROWS:
    raise ValueError(f"Unexpected product source shape: {raw.height} rows, columns={raw.columns}")
missing_required = {column: raw[column].null_count() for column in REQUIRED_COLUMNS if raw[column].null_count()}
if missing_required:
    raise ValueError(f"Required product values are NULL: {missing_required}")
if raw["sku"].n_unique() != raw.height:
    raise ValueError("Product SKU values are not unique in the source")
if set(raw["is_bundle"].unique().to_list()) != {"TRUE", "FALSE"}:
    raise ValueError("Unexpected is_bundle source values")
display(raw.head(5))


# 3. Cleaning & Typing

Use strings for product identifiers and descriptions, Decimal(18,2) for unit costs, and Boolean for the bundle flag.


In [ ]:
staged = raw.with_columns([
    pl.col("unit_cost_czk").cast(pl.Decimal(18, 2), strict=False),
    pl.col("is_bundle").str.to_uppercase().eq("TRUE").alias("is_bundle"),
])
if staged["unit_cost_czk"].null_count() != raw["unit_cost_czk"].null_count():
    raise ValueError("Invalid unit_cost_czk value found")
if staged["is_bundle"].null_count():
    raise ValueError("Invalid is_bundle value found")
if staged.schema["unit_cost_czk"] != pl.Decimal(18, 2) or staged.schema["is_bundle"] != pl.Boolean:
    raise TypeError("Unexpected product data types")
if staged.filter(pl.col("unit_cost_czk") < 0).height:
    raise ValueError("Product unit costs must not be negative")
if staged.filter(pl.col("is_bundle") & pl.col("unit_cost_czk").is_not_null()).height:
    raise ValueError("Bundle unit costs must remain NULL")
if staged.filter(~pl.col("is_bundle") & pl.col("unit_cost_czk").is_null()).height:
    raise ValueError("Non-bundle products must have a unit cost")
if staged.filter(pl.col("is_bundle") & pl.col("bundle_components").is_null()).height:
    raise ValueError("Bundle component definitions must be preserved")


# 4. Data Quality & Reconciliation

Validate source row count and unique SKU keys. Parse bundle component definitions and verify positive integer quantities, catalog references, and non-NULL component costs without expanding bundles.


In [ ]:
if staged.height != EXPECTED_ROWS or staged["sku"].n_unique() != EXPECTED_ROWS:
    raise ValueError("Product row count or SKU uniqueness validation failed")
if staged.filter(pl.col("is_bundle")).height != EXPECTED_BUNDLES:
    raise ValueError(f"Expected {EXPECTED_BUNDLES} bundles")
if staged.filter(pl.col("is_bundle") & pl.col("unit_cost_czk").is_null()).height != EXPECTED_BUNDLES:
    raise ValueError("NULL unit costs were not preserved for every bundle")
bundle_components = (
    staged.filter(pl.col("is_bundle"))
    .select(pl.col("sku").alias("bundle_sku"), "bundle_components")
    .with_columns(pl.col("bundle_components").str.split("|"))
    .explode("bundle_components")
    .with_columns(pl.col("bundle_components").str.split(":").alias("_parts"))
    .with_columns([
        pl.col("_parts").list.len().alias("_parts_count"),
        pl.col("_parts").list.get(0, null_on_oob=True).str.strip_chars().alias("component_sku"),
        pl.col("_parts").list.get(1, null_on_oob=True).str.strip_chars().alias("quantity_text"),
    ])
)
if bundle_components.filter(
    (pl.col("_parts_count") != 2)
    | pl.col("component_sku").is_null()
    | (pl.col("component_sku") == "")
    | pl.col("quantity_text").is_null()
    | (~pl.col("quantity_text").str.contains(r"^\d+$"))
).height:
    raise ValueError("Bundle definitions must use SKU:positive-integer entries separated by '|'")

bundle_components = bundle_components.with_columns(
    pl.col("quantity_text").cast(pl.Int64, strict=False).alias("component_quantity")
)
if bundle_components.filter(
    pl.col("component_quantity").is_null() | (pl.col("component_quantity") <= 0)
).height:
    raise ValueError("Bundle component quantities must be positive integers")

component_catalog = staged.select(
    pl.col("sku").alias("component_catalog_sku"),
    pl.col("unit_cost_czk").alias("component_unit_cost_czk"),
).with_columns(pl.lit(True).alias("_sku_found"))
component_checks = bundle_components.join(
    component_catalog,
    left_on="component_sku",
    right_on="component_catalog_sku",
    how="left",
)
if component_checks.filter(pl.col("_sku_found").is_null()).height:
    raise ValueError("Bundle component SKU is missing from the product catalog")
if component_checks.filter(pl.col("component_unit_cost_czk").is_null()).height:
    raise ValueError("Bundle components must have non-NULL unit costs")
display(pl.DataFrame({"rows": [staged.height], "unique_skus": [staged["sku"].n_unique()], "bundles": [staged.filter(pl.col('is_bundle')).height], "null_bundle_costs": [staged.filter(pl.col('is_bundle') & pl.col('unit_cost_czk').is_null()).height], "validated_component_entries": [bundle_components.height]}))
display(staged.head(5))


# 5. Parquet Export

Export the validated catalog without bundle expansion, then read it back and verify its schema and row count.


In [ ]:
staged.write_parquet(OUTPUT_PATH)
exported = pl.read_parquet(OUTPUT_PATH)
if exported.height != EXPECTED_ROWS or exported.schema != staged.schema:
    raise ValueError(f"Product Parquet export validation failed: {OUTPUT_PATH}")
display(pl.DataFrame({"output_file": [OUTPUT_PATH.name], "rows": [exported.height], "schema": [str(exported.schema)]}))
